# LLaVA-OneVision-2 8B Colab Server

OpenAI-compatible Colab server for construction accident VL experiments.

- Default model: `lmms-lab/LLaVA-OneVision-2-8B-ov`
- Endpoint: `/v1/chat/completions`
- Local FastAPI backend should use the printed `LLM_API_BASE`.
- Use only videos/images that you have permission to analyze.


In [ ]:
!pip install -q transformers accelerate fastapi uvicorn pyngrok nest_asyncio pillow sentencepiece protobuf


In [ ]:
import os, json, base64, re, threading, nest_asyncio
from io import BytesIO
from PIL import Image
import torch
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
import uvicorn
from transformers import AutoProcessor, AutoModelForImageTextToText

nest_asyncio.apply()


In [ ]:
MODEL_ID = os.environ.get("MODEL_ID", "lmms-lab/LLaVA-OneVision-2-8B-ov")

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
).eval()
print("loaded", MODEL_ID)


In [ ]:

SYSTEM_HINT = """You are a construction-site accident analysis vision-language model. Return JSON only.
Required fields: primary_type, secondary_type, injured_count, cause, confidence, timeline, workers, evidence, details, report_draft, prevention_actions.
primary_type must be one of: ??, ??, ??, ??.
Use only visible evidence. Do not infer legal liability, approval status, death, or training status.
Focus on accident cause: pre-accident action -> visual change -> accident result.
"""

def decode_data_url(value):
    if not value:
        return None
    if value.startswith('data:'):
        value = value.split(',', 1)[1]
    return Image.open(BytesIO(base64.b64decode(value))).convert('RGB')

def extract_prompt_and_image(messages):
    texts = []
    image = None
    for message in messages:
        content = message.get('content', '')
        if isinstance(content, str):
            texts.append(content)
            continue
        for item in content:
            if item.get('type') == 'text':
                texts.append(item.get('text', ''))
            elif item.get('type') == 'image_url':
                url = item.get('image_url', {}).get('url', '')
                if url.startswith('data:'):
                    image = decode_data_url(url)
    return '\n'.join(texts).strip(), image

def normalize_messages(messages):
    normalized = []
    for message in messages:
        content = []
        for item in message.get('content', []):
            if item.get('type') == 'text':
                content.append({'type': 'text', 'text': item.get('text', '')})
            elif item.get('type') == 'image_url':
                url = item.get('image_url', {}).get('url', '')
                if url.startswith('data:'):
                    content.append({'type': 'image', 'image': decode_data_url(url)})
        if not content:
            content = [{'type': 'text', 'text': SYSTEM_HINT}]
        normalized.append({'role': message.get('role', 'user'), 'content': content})
    if normalized:
        normalized[0]['content'].insert(0, {'type': 'text', 'text': SYSTEM_HINT})
    return normalized

def extract_json_text(text):
    text = str(text).strip()
    match = re.search(r'\{.*\}', text, re.S)
    return match.group(0) if match else text


In [ ]:
def run_model(messages, max_tokens=1000):
    prompt, image = extract_prompt_and_image(messages)
    if image is None:
        image = Image.new('RGB', (768, 768), 'white')
    conversation = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": SYSTEM_HINT + "\n" + prompt},
        ],
    }]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=image, text=text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return processor.decode(output[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)


In [ ]:

app = FastAPI(title='VL Colab OpenAI-Compatible Server')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_ID}

@app.post('/v1/chat/completions')
async def chat_completions(request: Request):
    body = await request.json()
    messages = body.get('messages', [])
    max_tokens = int(body.get('max_tokens') or 1000)
    answer = run_model(messages, max_tokens=max_tokens)
    answer = extract_json_text(answer)
    return {
        'id': 'construction-accident-vl-colab',
        'object': 'chat.completion',
        'model': MODEL_ID,
        'choices': [{
            'index': 0,
            'message': {'role': 'assistant', 'content': answer},
            'finish_reason': 'stop',
        }],
    }


In [ ]:

NGROK_TOKEN = os.environ.get('NGROK_AUTHTOKEN', '')
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000).public_url
print('LLM_API_BASE=' + public_url + '/v1')
thread = threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info'), daemon=True)
thread.start()
